# ULTIMATE PANDAS COOKBOOK: EXTENDED WITH JOINS, ADVANCED FILTERS & OPTIMIZATION

In [4]:
import numpy as np
import pandas as pd

In [6]:
# ------------------------------------------------------------------------------
# 1. SETUP RAW DATASET (WITH INTENTIONAL ANOMALIES & NULLS)
# ------------------------------------------------------------------------------
print("--- 1. INITIAL SEED DATASET ---")
raw_data = [
    (1, "techm", "2024-01-12", 60000.0, "DE", 4.1),
    (2, "epam", "2024-10-06", 85000.0, "DS", 3.8),
    (2, "google", "2025-09-19", 140000.0, "DE", 4.5),
    (3, "creditbank", "2023-09-14", 95000.0, "DA", 4.0),
    (4, "citi", "2024-09-19", 105000.0, "DE", 3.9),
    (1, "amazon", "2025-06-11", 130000.0, "DE", 4.6),
    (3, "wipro", "2024-08-29", 80000.0, "DA", 3.5),
    (4, "tcs", "2025-09-29", 110000.0, "DE", 4.2),
    (5, "google", "2026-07-05", 150000.0, "DS", 4.7),
    (5, "meta", "2026-07-15", 170000.0, "DS", 4.9),
    (6, "google", "2026-02-10", 155000.0, "DE", 4.8),
    (6, "google", "2026-02-10", 155000.0, "DE", 4.8),  # Duplicate
    (7, "startup-x", None, None, "DA", 3.0),  # Nulls
    (8, "legacy-co", "2022-03-04", 50000.0, None, None),  # Nulls
]

columns = ["emp_id", "company", "DOJ", "salary", "dept", "rating"]
df = pd.DataFrame(raw_data, columns=columns)
print(df)

--- 1. INITIAL SEED DATASET ---
    emp_id     company         DOJ    salary  dept  rating
0        1       techm  2024-01-12   60000.0    DE     4.1
1        2        epam  2024-10-06   85000.0    DS     3.8
2        2      google  2025-09-19  140000.0    DE     4.5
3        3  creditbank  2023-09-14   95000.0    DA     4.0
4        4        citi  2024-09-19  105000.0    DE     3.9
5        1      amazon  2025-06-11  130000.0    DE     4.6
6        3       wipro  2024-08-29   80000.0    DA     3.5
7        4         tcs  2025-09-29  110000.0    DE     4.2
8        5      google  2026-07-05  150000.0    DS     4.7
9        5        meta  2026-07-15  170000.0    DS     4.9
10       6      google  2026-02-10  155000.0    DE     4.8
11       6      google  2026-02-10  155000.0    DE     4.8
12       7   startup-x        None       NaN    DA     3.0
13       8   legacy-co  2022-03-04   50000.0  None     NaN


# DEDUPLICATION & NULL VALUE HANDLING

In [7]:
df = df.drop_duplicates()
df["salary"] = df["salary"].fillna(df["salary"].median()).astype(int)
df["dept"] = df["dept"].fillna("UNKNOWN").str.upper()
df["rating"] = df["rating"].fillna(df["rating"].mean().round(1))
df["DOJ"] = pd.to_datetime(df["DOJ"]).bfill()
df["company"] = df["company"].str.strip().str.upper()
print(df)

    emp_id     company        DOJ  salary     dept  rating
0        1       TECHM 2024-01-12   60000       DE     4.1
1        2        EPAM 2024-10-06   85000       DS     3.8
2        2      GOOGLE 2025-09-19  140000       DE     4.5
3        3  CREDITBANK 2023-09-14   95000       DA     4.0
4        4        CITI 2024-09-19  105000       DE     3.9
5        1      AMAZON 2025-06-11  130000       DE     4.6
6        3       WIPRO 2024-08-29   80000       DA     3.5
7        4         TCS 2025-09-29  110000       DE     4.2
8        5      GOOGLE 2026-07-05  150000       DS     4.7
9        5        META 2026-07-15  170000       DS     4.9
10       6      GOOGLE 2026-02-10  155000       DE     4.8
12       7   STARTUP-X 2022-03-04  107500       DA     3.0
13       8   LEGACY-CO 2022-03-04   50000  UNKNOWN     4.2


# NEW: DATA MERGING & JOINS (VLOOKUP / SQL JOIN Equivalent)

In [10]:
print("\n--- 3. JOIN / MERGE OPERATIONS ---")
# ating a secondary lookup dimension table
dept_lookup_data = [
    ("DE", "Data Engineering", "Tech Tower A"),
    ("DS", "Data Science", "Tech Tower B"),
    ("D", "Data Analytics", "Innovation Hub"),
]
df_dept_info = pd.DataFrame(
    dept_lookup_data, columns=["dept", "dept_full_name", "office_location"]
)

# forming an Left Join
df = pd.merge(df, df_dept_info, on="dept", how="left")
# lacing missing lookup values for the 'UNKNOWN' department row
df["dept_full_name"] = df["dept_full_name"].fillna("Unassigned Dept")
df["office_location"] = df["office_location"].fillna("Remote / Bench")
print(df[["emp_id", "company", "dept", "dept_full_name", "office_location"]].head(6))



--- 3. JOIN / MERGE OPERATIONS ---
   emp_id     company dept    dept_full_name office_location
0       1       TECHM   DE  Data Engineering    Tech Tower A
1       2        EPAM   DS      Data Science    Tech Tower B
2       2      GOOGLE   DE  Data Engineering    Tech Tower A
3       3  CREDITBANK   DA   Unassigned Dept  Remote / Bench
4       4        CITI   DE  Data Engineering    Tech Tower A
5       1      AMAZON   DE  Data Engineering    Tech Tower A


# ADVANCED STRING FILTERING (Wildcards & Substrings)

In [11]:
print("\n--- 4. STRING WILD CARD FILTERING (str.contains) ---")
# d all companies containing 'BANK' or 'CO' using regex flag or standard wildcards
wildcard_filter = df[
    df["company"].str.contains("BANK|CO|TECH", case=False, na=False)
]
print(wildcard_filter[["emp_id", "company", "dept"]])



--- 4. STRING WILD CARD FILTERING (str.contains) ---
    emp_id     company     dept
0        1       TECHM       DE
3        3  CREDITBANK       DA
12       8   LEGACY-CO  UNKNOWN


# RECURSIVE VALUE REPLACEMENT / TEXT MAPPING

In [12]:
print("\n--- 5. VALUE MAPPING AND REPLACEMENT ---")
# ap explicit values across a column smoothly
company_remap = {"TECHM": "TECH MAHINDRA", "TCS": "TATA CONSULTANCY SERVICES"}
df["company"] = df["company"].replace(company_remap)
print(df[["emp_id", "company"]].head(3))



--- 5. VALUE MAPPING AND REPLACEMENT ---
   emp_id        company
0       1  TECH MAHINDRA
1       2           EPAM
2       2         GOOGLE


# FEATURE ENGINEERING & TEMPORAL EXTRACTORS

In [14]:
conditions = [
    (df["salary"] >= 130000),
    (df["salary"] >= 90000) & (df["salary"] < 130000),
    (df["salary"] < 90000),
]
choices = ["Tier-1 (High)", "Tier-2 (Mid)", "Tier-3 (Entry)"]
df["salary_tier"] = np.select(conditions, choices, default="Unclassified")
df["performance_profile"] = df["rating"].apply(
    lambda r: "Elite" if r >= 4.5 else "Solid"
)

df["join_year"] = df["DOJ"].dt.year
df["join_month"] = df["DOJ"].dt.month
df["day_name"] = df["DOJ"].dt.day_name()
print(df)

    emp_id                    company        DOJ  salary     dept  rating  \
0        1              TECH MAHINDRA 2024-01-12   60000       DE     4.1   
1        2                       EPAM 2024-10-06   85000       DS     3.8   
2        2                     GOOGLE 2025-09-19  140000       DE     4.5   
3        3                 CREDITBANK 2023-09-14   95000       DA     4.0   
4        4                       CITI 2024-09-19  105000       DE     3.9   
5        1                     AMAZON 2025-06-11  130000       DE     4.6   
6        3                      WIPRO 2024-08-29   80000       DA     3.5   
7        4  TATA CONSULTANCY SERVICES 2025-09-29  110000       DE     4.2   
8        5                     GOOGLE 2026-07-05  150000       DS     4.7   
9        5                       META 2026-07-15  170000       DS     4.9   
10       6                     GOOGLE 2026-02-10  155000       DE     4.8   
11       7                  STARTUP-X 2022-03-04  107500       DA     3.0   

# ANALYTICAL WINDOW FUNCTIONS & ROW SEQUENCING

In [16]:
df["company_sequence_rank"] = df.groupby("emp_id")["DOJ"].rank(
    method="first", ascending=True
)
df = df.sort_values(by=["emp_id", "DOJ"])
df["previous_company_salary"] = df.groupby("emp_id")["salary"].shift(1)
df["salary_jump_amount"] = df["salary"] - df["previous_company_salary"]
print(df)

    emp_id                    company        DOJ  salary     dept  rating  \
0        1              TECH MAHINDRA 2024-01-12   60000       DE     4.1   
5        1                     AMAZON 2025-06-11  130000       DE     4.6   
1        2                       EPAM 2024-10-06   85000       DS     3.8   
2        2                     GOOGLE 2025-09-19  140000       DE     4.5   
3        3                 CREDITBANK 2023-09-14   95000       DA     4.0   
6        3                      WIPRO 2024-08-29   80000       DA     3.5   
4        4                       CITI 2024-09-19  105000       DE     3.9   
7        4  TATA CONSULTANCY SERVICES 2025-09-29  110000       DE     4.2   
8        5                     GOOGLE 2026-07-05  150000       DS     4.7   
9        5                       META 2026-07-15  170000       DS     4.9   
10       6                     GOOGLE 2026-02-10  155000       DE     4.8   
11       7                  STARTUP-X 2022-03-04  107500       DA     3.0   

# PERFORMANCE & MEMORY OPTIMIZATION

In [17]:
print("\n--- 8. MEMORY OPTIMIZATION (Before vs After Category Cast) ---")
print("Initial memory size (bytes):", df["salary_tier"].memory_usage(index=False))
# ncast object strings to Categorical elements for faster operations on low-cardinality strings
df["salary_tier"] = df["salary_tier"].astype("category")
df["dept"] = df["dept"].astype("category")
print("Optimized memory size (bytes):", df["salary_tier"].memory_usage(index=False))


--- 8. MEMORY OPTIMIZATION (Before vs After Category Cast) ---
Initial memory size (bytes): 104
Optimized memory size (bytes): 145


# FINAL PIPELINE STATE REPORT & DATA STAGING (EXPORT)

In [18]:
print("\n--- 9. FINAL PIPELINE STATE REPORT ---")
print(df.info())



--- 9. FINAL PIPELINE STATE REPORT ---
<class 'pandas.core.frame.DataFrame'>
Index: 13 entries, 0 to 12
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   emp_id                   13 non-null     int64         
 1   company                  13 non-null     object        
 2   DOJ                      13 non-null     datetime64[ns]
 3   salary                   13 non-null     int64         
 4   dept                     13 non-null     category      
 5   rating                   13 non-null     float64       
 6   dept_full_name_x         13 non-null     object        
 7   office_location_x        13 non-null     object        
 8   dept_full_name_y         9 non-null      object        
 9   office_location_y        9 non-null      object        
 10  dept_full_name           13 non-null     object        
 11  office_location          13 non-null     object        
 12  sal

In [20]:
# ging out structural layers (Commented out target executions)
df.to_csv("processed_employee_pipeline.csv", index=False)

In [21]:
df.to_json("processed_employee_pipeline.json", orient="records", date_format="iso")